# Dump validation predictions → `val_df.csv` (for error analysis)



In [ ]:
import os, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import timm
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from tqdm.auto import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| timm', timm.__version__)

In [ ]:
# ---------- config ----------
CACHE_DIR  = '/kaggle/input/datasets/shashaboii/panda-tiles-36x256/tiles_cache'
BACKBONE   = 'efficientnet_b0'
MODEL_NAME = 'effnetb0_36x256_fold0'
# set MODEL_PATH to your published checkpoint; the walk below helps you find it
MODEL_PATH = f'/kaggle/input/datasets/shashaboii/panda-effnetb0-fold0/effnetb0_36x256_fold0_best.pth'
fold = 0; SEED = 42
batch_size = 8; num_workers = 4

# helper: locate the .pth if the path above is wrong
if not os.path.exists(MODEL_PATH):
    print('checkpoint not at', MODEL_PATH, '- searching /kaggle/input ...')
    for r, _, fs in os.walk('/kaggle/input'):
        for f in fs:
            if f.endswith('_best.pth'): print('  found:', os.path.join(r, f))

In [ ]:
# rebuild the SAME dataframe + fold split as training
try:
    df = pd.read_csv(os.path.join(CACHE_DIR, 'train.csv'))
except Exception:
    df = pd.read_csv('../input/prostate-cancer-grade-assessment/train.csv')
have = set(f[:-4] for f in os.listdir(CACHE_DIR) if f.endswith('.jpg'))
df = df[df.image_id.isin(have)].reset_index(drop=True)

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
df['fold'] = -1
for i, (_, va) in enumerate(skf.split(df, df.isup_grade)):
    df.loc[va, 'fold'] = i
df_valid = df[df.fold == fold].reset_index(drop=True)
print('validation slides:', len(df_valid))

In [ ]:
class ValDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        m = np.array(Image.open(os.path.join(CACHE_DIR, f'{row.image_id}.jpg')))
        m = 255 - m                                    # same inversion as training
        m = (m.astype(np.float32) / 255).transpose(2, 0, 1)
        return torch.tensor(m)

class enetv2(nn.Module):
    def __init__(self, backbone=BACKBONE, out_dim=5):
        super().__init__()
        self.enet = timm.create_model(backbone, pretrained=False, num_classes=0, global_pool='avg')
        self.myfc = nn.Linear(self.enet.num_features, out_dim)
    def forward(self, x): return self.myfc(self.enet(x))

model = enetv2().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu')); model.eval()
print('loaded', MODEL_PATH)

In [ ]:
loader = DataLoader(ValDataset(df_valid), batch_size=batch_size, num_workers=num_workers, shuffle=False)
preds = []
with torch.no_grad():
    for data in tqdm(loader, desc='predicting val'):
        logits = model(data.to(device))
        preds.append(logits.sigmoid().sum(1).round().clamp(0, 5).cpu())
preds = torch.cat(preds).int().numpy()

val_df = df_valid[['image_id', 'data_provider', 'isup_grade']].copy()
val_df = val_df.rename(columns={'isup_grade': 'true_isup'})
val_df['pred_isup'] = preds
val_df.to_csv('/kaggle/working/val_df_effnet.csv', index=False)
print('saved val_df_effnet.csv', val_df.shape)
val_df.head()

In [ ]:
# sanity check — should match the val QWK your training run reported
qwk = cohen_kappa_score(val_df.true_isup, val_df.pred_isup, weights='quadratic')
print('validation QWK:', round(qwk, 4))
for prov in val_df.data_provider.unique():
    s = val_df[val_df.data_provider == prov]
    print(f'  {prov:11s} QWK {cohen_kappa_score(s.true_isup, s.pred_isup, weights="quadratic"):.4f} (n={len(s)})')
print('\nconfusion matrix (rows=true, cols=pred):')
print(confusion_matrix(val_df.true_isup, val_df.pred_isup, labels=list(range(6))))